# 02 - Scalar Graph

这一节只从**使用者视角**理解 `Value`。

这一节不读源码，只观察 `Value` 的使用效果。内部实现放到下一节 `03_value_implementation.ipynb`。

这一节只回答三个问题：

1. 用 `Value` 写一个公式，前向计算会得到什么？
2. `backward()` 之后，`grad` 变成什么？
3. 这些 `grad` 和我们手算的偏导是否一致？

我们继续使用这个例子：

$$
L = 2(ab + a)
$$

取值：

$$
a=2,\quad b=3
$$


<!-- xiao-preview:start -->
## 课前预习作业：先手算，再用 Value 验证

这一节不看源码，只把 `Value` 当成一个会记录梯度的数字。先手算下面三个值，后面用 `L.backward()` 验证。


In [1]:
# TODO: 把 None 改成你手算出来的数字。
# a=2, b=3, L = 2 * (a*b + a)
preview_L_data = None
preview_a_grad = None
preview_b_grad = None


def qa_check_02_preview(L_data, a_grad, b_grad):
    values = [L_data, a_grad, b_grad]
    if any(v is None for v in values):
        print('请先填写 TODO：先手算 L.data、a.grad、b.grad。')
        return
    assert L_data == 16, f'L.data 应该是 16，但你填的是 {L_data}'
    assert a_grad == 8, f'a.grad 应该是 8，但你填的是 {a_grad}'
    assert b_grad == 4, f'b.grad 应该是 4，但你填的是 {b_grad}'
    print('OK: 预习通过，下一步用 Value 跑同一个例子。')


qa_check_02_preview(preview_L_data, preview_a_grad, preview_b_grad)


请先填写 TODO：先手算 L.data、a.grad、b.grad。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

先正向：`a*b=6`，`a*b+a=8`，`L=16`。  
再反向：`dL/da=2(b+1)`，`dL/db=2a`。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_L_data = 16
preview_a_grad = 8
preview_b_grad = 4
```

</details>


## 怎么学这本

建议顺序：

```text
先运行代码 -> 看输出 -> 回到公式手算 -> 再看输出是否一致
```

本节过关标准：

```text
1. 能说出 a、b、c、d、L 的 data 是多少
2. 能说出 backward 前 grad 为什么都是 0
3. 能说出 backward 后 a.grad=8、b.grad=4 分别代表什么
4. 能把 a.grad 和 dL/da 对上，把 b.grad 和 dL/db 对上
```

暂时不用研究内部实现。先把这一节当成一次实验：输入公式，观察前向结果和反向梯度。


## 本节不要研究源码

这一节只练使用层心智模型：

```text
Value 数字 -> 前向算出 L.data -> L.backward() -> 看 a.grad / b.grad
```

看到 `_prev`、`_op`、`_backward` 时，只把它们当成观察信息。真正源码解释在下一节。

## 0. Setup

导入本地 `micrograd`。如果你从 Jupyter 打开这本 notebook，直接运行即可。


In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))
from micrograd.engine import Value

print('project root:', PROJECT_ROOT)



def assert_close(actual, expected, name):
    assert abs(actual - expected) < 1e-9, f'{name}: expected {expected}, got {actual}'
    print(f'OK: {name} = {actual}')



def qa_check_square_sum_graph(a, b, L):
    if L is None:
        print('TODO: 请先把 L = None 改成 Value 公式，比如由 a 和 b 算出 L。')
        return

    L.backward()
    expected_L = (a.data + b.data) ** 2
    expected_grad = 2 * (a.data + b.data)
    assert_close(L.data, expected_L, 'L.data')
    assert_close(a.grad, expected_grad, 'a.grad')
    assert_close(b.grad, expected_grad, 'b.grad')
    print('OK: square-sum graph passed.')


project root: /Users/barry/IdeaProjects/llm/micrograd


## 1. Forward: Build A Tiny Formula

把公式拆成几步：

$$
c = ab
$$

$$
d = c + a
$$

$$
L = 2d
$$

对应代码：


### Predict Before Run

先别运行下一格。你先写下这 5 个值：

```text
a.data = ?
b.data = ?
c.data = a*b = ?
d.data = c+a = ?
L.data = d*2 = ?
```

写完再运行。

In [3]:
a = Value(2.0)
b = Value(3.0)
c = a * b
d = c + a
L = d * 2

print('a =', a)
print('b =', b)
print('c = a * b =', c)
print('d = c + a =', d)
print('L = d * 2 =', L)


a = Value(data=2.0, grad=0)
b = Value(data=3.0, grad=0)
c = a * b = Value(data=6.0, grad=0)
d = c + a = Value(data=8.0, grad=0)
L = d * 2 = Value(data=16.0, grad=0)


你现在先只看 `data`：

```text
a.data = 2
b.data = 3
c.data = 6
 d.data = 8
L.data = 16
```

这就是前向计算。

注意：这里的 `Value(data=6.0, grad=0)` 表示这个节点当前数值是 6，梯度还没计算，所以 `grad=0`。


## 2. A Small Printer For Nodes

下面这个 `show_all` 只是为了把每个节点打印得更清楚。

你不用纠结它的 Python 写法，只需要看输出里的四列：

```text
name     节点名字
data     当前数值
grad     当前梯度
parents  这个节点由哪些节点算出来
```


In [4]:
nodes = {'a': a, 'b': b, 'c': c, 'd': d, 'L': L}


def value_name(value, nodes):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'


def show(name, value, nodes):
    parents = [value_name(parent, nodes) for parent in value._prev]
    print(f'{name:>2} | data={value.data:>5} | grad={value.grad:>5} | parents={parents}')


def show_all(title, nodes):
    print(title)
    for name in ['a', 'b', 'c', 'd', 'L']:
        show(name, nodes[name], nodes)


show_all('Before backward', nodes)


Before backward
 a | data=  2.0 | grad=    0 | parents=[]
 b | data=  3.0 | grad=    0 | parents=[]
 c | data=  6.0 | grad=    0 | parents=['a', 'b']
 d | data=  8.0 | grad=    0 | parents=['c', 'a']
 L | data= 16.0 | grad=    0 | parents=['d', 'Value(2)']


观察输出：

```text
a, b 是最初给的数，所以 parents=[]
c 来自 a 和 b
d 来自 c 和 a
L 来自 d 和普通数字 2
```

这就是“计算图”的意思：每个中间结果都能追溯自己从哪里来。

这本只要求你能看懂这个现象；至于它源码怎么记录，下一节再学。


## 3. Backward: Ask How L Changes With a And b

现在调用：

```python
L.backward()
```

它会自动计算：

$$
\frac{\partial L}{\partial a}
$$

和：

$$
\frac{\partial L}{\partial b}
$$

也就是：

```text
a.grad
b.grad
```


In [5]:
L.backward()
show_all('After backward', nodes)


After backward
 a | data=  2.0 | grad=  8.0 | parents=[]
 b | data=  3.0 | grad=  4.0 | parents=[]
 c | data=  6.0 | grad=    2 | parents=['a', 'b']
 d | data=  8.0 | grad=    2 | parents=['c', 'a']
 L | data= 16.0 | grad=    1 | parents=['d', 'Value(2)']


重点看最后：

```text
a.grad = 8
b.grad = 4
```

含义是：

```text
a 稍微变大一点，L 大约按 8 倍速度变化
b 稍微变大一点，L 大约按 4 倍速度变化
```

也就是：

$$
a.grad = \frac{\partial L}{\partial a}
$$

$$
b.grad = \frac{\partial L}{\partial b}
$$


## 4. Check With Hand Math

原式：

$$
L = 2(ab+a)
$$

对 $a$ 求偏导，把 $b$ 当常数：

$$
\frac{\partial L}{\partial a} = 2(b+1)
$$

对 $b$ 求偏导，把 $a$ 当常数：

$$
\frac{\partial L}{\partial b} = 2a
$$

代入 $a=2,b=3$：

$$
\frac{\partial L}{\partial a}=2(3+1)=8
$$

$$
\frac{\partial L}{\partial b}=2\times2=4
$$

和 `micrograd` 输出一致。


In [6]:
expected_da = 2 * (3 + 1)
expected_db = 2 * 2

print('hand math dL/da =', expected_da)
print('micrograd a.grad =', a.grad)
print()
print('hand math dL/db =', expected_db)
print('micrograd b.grad =', b.grad)


hand math dL/da = 8
micrograd a.grad = 8.0

hand math dL/db = 4
micrograd b.grad = 4.0


## 5. Numerical Check

再用“微小扰动”验证一下。

如果把 $a$ 稍微加一点点，看看 $L$ 增加多少，也能近似得到导数：

$$
\frac{\partial L}{\partial a}
\approx
\frac{f(a+h,b)-f(a-h,b)}{2h}
$$


In [7]:
def f(a_value, b_value):
    return 2 * (a_value * b_value + a_value)

h = 1e-6
num_da = (f(2.0 + h, 3.0) - f(2.0 - h, 3.0)) / (2 * h)
num_db = (f(2.0, 3.0 + h) - f(2.0, 3.0 - h)) / (2 * h)

print('numerical dL/da =', num_da)
print('micrograd a.grad =', a.grad)
print()
print('numerical dL/db =', num_db)
print('micrograd b.grad =', b.grad)


numerical dL/da = 8.000000000230045
micrograd a.grad = 8.0

numerical dL/db = 4.000000000559112
micrograd b.grad = 4.0


## 6. Mini Lab: 改一行，重新跑一遍

下面把同一个公式封装成函数。你可以改 `examples` 里的数字，观察 `data` 和 `grad` 怎么变。

In [8]:
def run_value_example(a_data, b_data):
    a = Value(a_data)
    b = Value(b_data)
    L = 2 * (a * b + a)
    L.backward()
    expected_da = 2 * (b_data + 1)
    expected_db = 2 * a_data
    print(f'a={a_data:>5}, b={b_data:>5}, L.data={L.data:>6}, a.grad={a.grad:>6}, b.grad={b.grad:>6}')
    assert_close(a.grad, expected_da, 'a.grad')
    assert_close(b.grad, expected_db, 'b.grad')
    print()

examples = [(2.0, 3.0), (4.0, -2.0), (0.0, 5.0)]
for a_data, b_data in examples:
    run_value_example(a_data, b_data)

a=  2.0, b=  3.0, L.data=  16.0, a.grad=   8.0, b.grad=   4.0
OK: a.grad = 8.0
OK: b.grad = 4.0

a=  4.0, b= -2.0, L.data=  -8.0, a.grad=  -2.0, b.grad=   8.0
OK: a.grad = -2.0
OK: b.grad = 8.0

a=  0.0, b=  5.0, L.data=   0.0, a.grad=  12.0, b.grad=   0.0
OK: a.grad = 12.0
OK: b.grad = 0.0



## 7. TODO Lab - 换一个公式

现在换成一个更简单的公式：

$$
L = (a+b)^2
$$

TODO：自己新建 cell，写出 `Value` 版本，然后检查：

```text
a=1, b=2 时，a.grad 和 b.grad 应该都是 6
```

下面是参考检查。先自己写，再运行。

In [9]:
a = Value(1.0)
b = Value(2.0)

# TODO: 把 None 改成 Value 公式。
L = None

qa_check_square_sum_graph(a, b, L)

TODO: 请先把 L = None 改成 Value 公式，比如由 a 和 b 算出 L。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

先写 `a + b`，再整体平方。micrograd 里平方可以写成 `(...) ** 2`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
a = Value(1.0)
b = Value(2.0)
L = (a + b) ** 2
qa_check_square_sum_graph(a, b, L)
```

</details>

## 8. Important Notebook Detail

如果你重复运行同一个 `L.backward()` cell，梯度可能会累加。

学习时最简单的做法：每次重新从这里开始建图：

```python
a = Value(2.0)
b = Value(3.0)
c = a * b
d = c + a
L = d * 2
```

后面源码部分会解释为什么 micrograd 用 `+=` 累加梯度。


## 9. What To Remember

这一节只记住这几句话：

```text
1. 前向计算：用 Value 算出 L.data
2. 反向传播：调用 L.backward()
3. backward 后，a.grad 就是 dL/da，b.grad 就是 dL/db
4. micrograd 算出来的 grad 和我们手算偏导一致
```

下一节 `03_value_implementation.ipynb` 再研究它内部是怎么做到的。


<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 4 条自检：

```text
1. 我能从 a、b、c、d、L 讲清楚前向计算顺序。
2. 我能解释 L.backward() 后 a.grad 和 b.grad 的含义。
3. 我能把 micrograd 的输出和手算偏导对上。
4. 我没有在这一节纠结源码实现，源码留给 03。
```


## AI 复盘检查 Prompt

把下面这段发送给任意 AI，让它检查你是否理解 `Value` 的使用效果：

```text
你是我的 micrograd Value 使用层学习检查官。

我刚学完一节内容，主题是：不看源码，只从使用者视角观察 Value 的前向计算和 backward 后的 grad。

本节例子是：
a = Value(2.0)
b = Value(3.0)
c = a * b
d = c + a
L = d * 2
也就是 L = 2(ab+a)。

请你用一问一答的方式检查我。规则：
1. 一次只问一个问题，等我回答后再评价。
2. 不要问源码实现细节，比如 __add__、_backward、拓扑排序，这些属于下一节。
3. 如果我回答不清楚，请让我回到 data、grad、偏导这三个词上解释。
4. 最后给我一个 0-100 的掌握度评分，并判断我是否可以进入 Value 源码实现。

请按这个顺序检查我：
1. 前向计算后，a、b、c、d、L 的 data 分别是多少？
2. backward 前，为什么这些节点的 grad 都是 0？
3. 调用 L.backward() 后，a.grad 和 b.grad 分别是多少？
4. a.grad = 8 这句话在数学上是什么意思？
5. b.grad = 4 这句话在数学上是什么意思？
6. micrograd 算出的 grad 和手算偏导如何对应？
7. 如果重复运行同一个 L.backward() cell，为什么结果可能让人困惑？

现在从第 1 个问题开始问我。
```
